In [ ]:
import requests
import pandas as pd
from dotenv import load_dotenv
import os
import time

load_dotenv()

TMDB_KEY = os.getenv("TMDB_API_KEY")
OMDB_KEY = os.getenv("OMDB_API_KEY")

print("✅ Ready")

✅ Ready


In [39]:
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import os

In [6]:
def get_movies_by_language(language, region, years=[2023, 2024, 2025]):
    movies = []
    
    for year in years:
        print(f"  Fetching {region} movies from {year}...")
        
        for page in range(1, 6):  # 5 pages = 100 movies
            url = f"https://api.themoviedb.org/3/discover/movie"
            params = {
                "api_key":               TMDB_KEY,
                "with_original_language": language,
                "primary_release_year":  year,
                "sort_by":               "popularity.desc",
                "page":                  page
            }
            r = requests.get(url, params=params).json()
            
            if not r.get("results"):
                break
            
            for movie in r["results"]:
                movies.append({
                    "title":   movie["title"],
                    "year":    year,
                    "region":  region,
                    "tmdb_id": movie["id"],
                    "popularity": movie.get("popularity", 0)
                })
            time.sleep(0.1)
    
    return movies

# Fetch all industries
all_movies = []

industries = [
    ("en", "Hollywood"),
    ("hi", "Bollywood"),
    ("ta", "Kollywood"),
    ("te", "Tollywood"),
    ("ml", "Mollywood"),
]

for lang, region in industries:
    print(f"\n🎬 Fetching {region}...")
    movies = get_movies_by_language(lang, region)
    all_movies.extend(movies)
    print(f"  → {len(movies)} movies found")

movies_df = pd.DataFrame(all_movies).drop_duplicates(subset="tmdb_id")
print(f"\n✅ Total: {len(movies_df)} movies across all industries")
movies_df.head(10)


🎬 Fetching Hollywood...
  Fetching Hollywood movies from 2023...
  Fetching Hollywood movies from 2024...
  Fetching Hollywood movies from 2025...
  → 300 movies found

🎬 Fetching Bollywood...
  Fetching Bollywood movies from 2023...
  Fetching Bollywood movies from 2024...
  Fetching Bollywood movies from 2025...
  → 300 movies found

🎬 Fetching Kollywood...
  Fetching Kollywood movies from 2023...
  Fetching Kollywood movies from 2024...
  Fetching Kollywood movies from 2025...
  → 300 movies found

🎬 Fetching Tollywood...
  Fetching Tollywood movies from 2023...
  Fetching Tollywood movies from 2024...
  Fetching Tollywood movies from 2025...
  → 300 movies found

🎬 Fetching Mollywood...
  Fetching Mollywood movies from 2023...
  Fetching Mollywood movies from 2024...
  Fetching Mollywood movies from 2025...
  → 300 movies found

✅ Total: 1484 movies across all industries


,title,year,region,tmdb_id,popularity
0,Barbie,2023,Hollywood,346698,27.9480
1,Scream VI,2023,Hollywood,934433,26.9248
2,Oppenheimer,2023,Hollywood,872585,25.7640
3,Fast X,2023,Hollywood,385687,25.3642
4,Spider-Man: Across the Spider-Verse,2023,Hollywood,569094,22.9215
5,Migration,2023,Hollywood,940551,19.0857
6,The Super Mario Bros. Movie,2023,Hollywood,502356,18.4874
7,Wonka,2023,Hollywood,787699,16.3243
8,Nutcracker Massacre,2023,Hollywood,981323,14.3144
9,John Wick: Chapter 4,2023,Hollywood,603692,13.5499


In [7]:


#get imdb from tmdb
def get_imdb_id(tmdb_id):
    url = f"https://api.themoviedb.org/3/movie/{tmdb_id}/external_ids"
    params = {"api_key": TMDB_KEY}
    r = requests.get(url, params=params).json()
    return r.get("imdb_id")

print("Getting IMDb IDs from TMDB...")
imdb_ids = []

for i, row in movies_df.iterrows():
    imdb_id = get_imdb_id(row["tmdb_id"])
    imdb_ids.append(imdb_id)
    if i % 50 == 0:
        print(f"  → {i}/{len(movies_df)} done")
    time.sleep(0.1)

movies_df["imdb_id"] = imdb_ids
movies_df = movies_df[movies_df["imdb_id"].notna()]
print(f"✅ {len(movies_df)} movies with valid IMDb IDs")

Getting IMDb IDs from TMDB...
  → 0/1484 done
  → 50/1484 done
  → 100/1484 done
  → 150/1484 done
  → 200/1484 done
  → 250/1484 done
  → 300/1484 done
  → 350/1484 done
  → 400/1484 done
  → 450/1484 done
  → 500/1484 done
  → 550/1484 done
  → 600/1484 done
  → 650/1484 done
  → 700/1484 done
  → 750/1484 done
  → 800/1484 done
  → 850/1484 done
  → 900/1484 done
  → 950/1484 done
  → 1000/1484 done
  → 1050/1484 done
  → 1100/1484 done
  → 1150/1484 done
  → 1200/1484 done
  → 1250/1484 done
  → 1300/1484 done
  → 1350/1484 done
  → 1400/1484 done
  → 1450/1484 done
✅ 1381 movies with valid IMDb IDs


In [24]:
def get_omdb_ratings(imdb_id):
    url = f"http://www.omdbapi.com/?i={imdb_id}&apikey={OMDB_KEY}"
    r = requests.get(url).json()
    
    if r.get("Response") == "False":
        return None, None, None
    
    imdb = r.get("imdbRating")
    imdb = float(imdb) if imdb and imdb != "N/A" else None
    
    rt = next((x["Value"] for x in r.get("Ratings", [])
               if x["Source"] == "Rotten Tomatoes"), None)
    
    genre = r.get("Genre")
    
    return imdb, rt, genre

print("Fetching IMDb + RT ratings from OMDB...")
imdb_ratings, rt_scores, genres = [], [], []

for i, row in movies_df.iterrows():
    imdb, rt, genre = get_omdb_ratings(row["imdb_id"])
    imdb_ratings.append(imdb)
    rt_scores.append(rt)
    genres.append(genre)
    
    if i % 20 == 0:
        print(f"  → {i}/{len(movies_df)} | {row['title']} | IMDb: {imdb} | RT: {rt}")
    time.sleep(0.2)

movies_df["imdb_rating"] = imdb_ratings
movies_df["rt_score"]    = rt_scores
movies_df["genre"]       = genres

# Keep only movies with valid ratings
movies_df = movies_df[movies_df["imdb_rating"].notna()]
print(f"\n✅ {len(movies_df)} movies with valid IMDb ratings")
movies_df.to_csv("data/movies_master.csv", index=False)
movies_df.head(10)

Fetching IMDb + RT ratings from OMDB...
  → 0/1381 | Barbie | IMDb: 6.8 | RT: 88%
  → 20/1381 | Killers of the Flower Moon | IMDb: 7.5 | RT: 93%
  → 40/1381 | Far Haven | IMDb: 5.8 | RT: None
  → 60/1381 | Dream Moms | IMDb: 5.8 | RT: None
  → 80/1381 | Broken Ties | IMDb: None | RT: None
  → 100/1381 | Música | IMDb: 6.4 | RT: None
  → 140/1381 | The Fall Guy | IMDb: 6.8 | RT: 82%
  → 160/1381 | Abigail | IMDb: 6.5 | RT: 83%
  → 180/1381 | Desire Lines | IMDb: 7.3 | RT: 82%
  → 200/1381 | Turbulence | IMDb: 4.8 | RT: None
  → 220/1381 | KPop Demon Hunters | IMDb: 7.5 | RT: 95%
  → 240/1381 | Atropia | IMDb: None | RT: 42%
  → 260/1381 | The Naked Gun | IMDb: 6.5 | RT: 87%
  → 280/1381 | Playdate | IMDb: 5.5 | RT: 23%
  → 300/1381 | Anu | IMDb: None | RT: None
  → 320/1381 | Kennedy | IMDb: 6.2 | RT: 56%
  → 380/1381 | Bloody Daddy | IMDb: 6.5 | RT: None
  → 400/1381 | Hisaab Barabar | IMDb: 5.8 | RT: None
  → 420/1381 | Stree 2 | IMDb: 6.9 | RT: 57%
  → 440/1381 | Jigra | IMDb: 6.1 | 

,title,year,region,tmdb_id,popularity,imdb_id,imdb_rating,rt_score,genre
0,Barbie,2023,Hollywood,346698,27.9480,tt1517268,6.8,88%,"Adventure, Comedy, Fantasy"
1,Scream VI,2023,Hollywood,934433,26.9248,tt17663992,6.4,77%,"Horror, Mystery, Thriller"
2,Oppenheimer,2023,Hollywood,872585,25.7640,tt15398776,8.2,93%,"Biography, Drama, History"
3,Fast X,2023,Hollywood,385687,25.3642,tt5433140,5.7,56%,"Action, Adventure, Crime"
4,Spider-Man: Across the Spider-Verse,2023,Hollywood,569094,22.9215,tt9362722,8.5,95%,"Animation, Action, Adventure"
5,Migration,2023,Hollywood,940551,19.0857,tt6495056,6.6,73%,"Animation, Adventure, Comedy"
6,The Super Mario Bros. Movie,2023,Hollywood,502356,18.4874,tt6718170,7.0,59%,"Animation, Adventure, Comedy"
7,Wonka,2023,Hollywood,787699,16.3243,tt6166392,6.9,82%,"Adventure, Comedy, Family"
8,Nutcracker Massacre,2023,Hollywood,981323,14.3144,tt16475076,3.3,None,"Horror, Thriller"
9,John Wick: Chapter 4,2023,Hollywood,603692,13.5499,tt10366206,7.6,94%,"Action, Crime, Thriller"


In [25]:
'''# Test OMDB key
import requests
from dotenv import load_dotenv
import os

load_dotenv()


print(f"Key loaded: {OMDB_KEY}")

# Test with a known movie
r = requests.get(f"http://www.omdbapi.com/?t=Barbie&apikey={OMDB_KEY}").json()
print(r)'''

'# Test OMDB key\nimport requests\nfrom dotenv import load_dotenv\nimport os\n\nload_dotenv()\n\n\nprint(f"Key loaded: {OMDB_KEY}")\n\n# Test with a known movie\nr = requests.get(f"http://www.omdbapi.com/?t=Barbie&apikey={OMDB_KEY}").json()\nprint(r)'

In [27]:
# Check actual column names first
print(movies_df.columns.tolist())

['title', 'year', 'region', 'tmdb_id', 'popularity', 'imdb_id', 'imdb_rating', 'rt_score', 'genre']


In [29]:
# Show movies where we have all 3 ratings
complete = movies_df[
    movies_df["imdb_rating"].notna() & 
    movies_df["rt_score"].notna()
].copy()

print(f"✅ Complete data for {len(complete)} movies\n")
complete[["title", "region", "imdb_rating", "rt_score", "genre"]].head(300)

✅ Complete data for 295 movies



,title,region,imdb_rating,rt_score,genre
0,Barbie,Hollywood,6.8,88%,"Adventure, Comedy, Fantasy"
1,Scream VI,Hollywood,6.4,77%,"Horror, Mystery, Thriller"
2,Oppenheimer,Hollywood,8.2,93%,"Biography, Drama, History"
3,Fast X,Hollywood,5.7,56%,"Action, Adventure, Crime"
4,Spider-Man: Across the Spider-Verse,Hollywood,8.5,95%,"Animation, Action, Adventure"
...,...,...,...,...,...
735,Rathnam,Kollywood,5.4,20%,Action
742,Siren,Kollywood,6.4,0%,"Action, Crime, Thriller"
751,Aranmanai 4,Kollywood,5.0,40%,Horror
776,Lal Salaam,Kollywood,4.5,40%,"Action, Drama, Sport"


In [44]:

from snowflake.connector.pandas_tools import write_pandas
import snowflake.connector

conn = snowflake.connector.connect(
    user="ATHULYA2303",
    password="Athulyabiju@2303",
    account="gsc07824.us-east-1",
    warehouse="COMPUTE_WH",
    database="MOVIE_ANALYTICS",
    schema="RAW"
)

# First create table in Snowflake if not exists
cursor = conn.cursor()
cursor.execute("""
    CREATE TABLE IF NOT EXISTS raw.movies_master (
        tmdb_id      STRING,
        imdb_id      STRING,
        title        STRING,
        year         STRING,
        genre        STRING,
        region       STRING,
        imdb_rating  FLOAT,
        rt_score     STRING,
        popularity   FLOAT
    )
""")

push_df = movies_df[[
    "tmdb_id", "imdb_id", "title", "year",
    "genre", "region", "imdb_rating", "rt_score", "popularity"
]].reset_index(drop=True)

push_df.columns = [c.upper() for c in push_df.columns]
push_df["TMDB_ID"] = push_df["TMDB_ID"].astype(str)
push_df["YEAR"] = push_df["YEAR"].astype(str)

write_pandas(conn, push_df, "MOVIES_MASTER",
             database="MOVIE_ANALYTICS", schema="RAW")
print(f"✅ {len(push_df)} movies pushed to Snowflake!")
conn.close()


✅ 571 movies pushed to Snowflake!


In [42]:
# Re-fetch Malayalam movies only
def get_movies_by_language(language, region, years=[2023, 2024, 2025]):
    movies = []
    for year in years:
        for page in range(1, 6):
            url = "https://api.themoviedb.org/3/discover/movie"
            params = {
                "api_key": "a2fcaa4082bd07085c705321c2b8b6e8",
                "with_original_language": language,
                "primary_release_year": year,
                "sort_by": "popularity.desc",
                "page": page
            }
            r = requests.get(url, params=params).json()
            if not r.get("results"):
                break
            for movie in r["results"]:
                movies.append({
                    "title":      movie["title"],
                    "year":       year,
                    "region":     region,
                    "tmdb_id":    movie["id"],
                    "popularity": movie.get("popularity", 0)
                })
            time.sleep(0.1)
    return movies

print("Fetching Mollywood movies...")
mollywood = get_movies_by_language("ml", "Mollywood")
mollywood_df = pd.DataFrame(mollywood)
print(f"✅ Found {len(mollywood_df)} Malayalam movies")
mollywood_df.head()

Fetching Mollywood movies...
✅ Found 300 Malayalam movies


,title,year,region,tmdb_id,popularity
0,Khali Purse of Billionaires,2023,Mollywood,880035,9.5568
1,King of Kotha,2023,Mollywood,855117,4.1451
2,Lovefully Yours Veda,2023,Mollywood,915388,3.8201
3,Imbam,2023,Mollywood,1044787,3.8105
4,Dance Party,2023,Mollywood,1089230,3.7674


In [33]:
# Step 1 - Get IMDb IDs
print("Getting IMDb IDs...")
imdb_ids = []
for i, row in mollywood_df.iterrows():
    url = f"https://api.themoviedb.org/3/movie/{row['tmdb_id']}/external_ids"
    r = requests.get(url, params={"api_key": "a2fcaa4082bd07085c705321c2b8b6e8"}).json()
    imdb_ids.append(r.get("imdb_id"))
    time.sleep(0.1)

mollywood_df["imdb_id"] = imdb_ids
mollywood_df = mollywood_df[mollywood_df["imdb_id"].notna()]
print(f"✅ {len(mollywood_df)} movies with IMDb IDs")

# Step 2 - Get OMDB ratings
print("\nGetting IMDb + RT ratings...")
imdb_ratings, rt_scores, genres = [], [], []

for i, row in mollywood_df.iterrows():
    url = f"http://www.omdbapi.com/?i={row['imdb_id']}&apikey=13acfb9e"
    r = requests.get(url).json()
    
    imdb = r.get("imdbRating")
    imdb = float(imdb) if imdb and imdb != "N/A" else None
    rt = next((x["Value"] for x in r.get("Ratings", [])
               if x["Source"] == "Rotten Tomatoes"), None)
    genre = r.get("Genre")
    
    imdb_ratings.append(imdb)
    rt_scores.append(rt)
    genres.append(genre)
    time.sleep(0.2)

mollywood_df["imdb_rating"] = imdb_ratings
mollywood_df["rt_score"]    = rt_scores
mollywood_df["genre"]       = genres

mollywood_df = mollywood_df[mollywood_df["imdb_rating"].notna()]
print(f"✅ {len(mollywood_df)} Mollywood movies with valid ratings")
mollywood_df.head(10)

Getting IMDb IDs...
✅ 269 movies with IMDb IDs

Getting IMDb + RT ratings...
✅ 0 Mollywood movies with valid ratings


,title,year,region,tmdb_id,popularity,imdb_id,imdb_rating,rt_score,genre


In [34]:
print(f"Total Malayalam movies: {len(mollywood_df)}")
print(f"With IMDb ID: {mollywood_df['imdb_id'].notna().sum()}")
print(f"With IMDb rating: {mollywood_df['imdb_rating'].notna().sum()}")

# Show sample
mollywood_df[["title", "imdb_id", "imdb_rating"]].head(10)

Total Malayalam movies: 0
With IMDb ID: 0
With IMDb rating: 0


,title,imdb_id,imdb_rating


In [35]:
# Search for known big Malayalam movies directly
big_movies = ["Manjummel Boys", "Premalu", "Aavesham", "Amaran", "Bramayugam"]

for movie in big_movies:
    r = requests.get(f"http://www.omdbapi.com/?t={movie}&apikey=13acfb9e").json()
    print(f"{movie} → IMDb: {r.get('imdbRating')} | Response: {r.get('Response')}")

Manjummel Boys → IMDb: None | Response: False
Premalu → IMDb: None | Response: False
Aavesham → IMDb: None | Response: False
Amaran → IMDb: None | Response: False
Bramayugam → IMDb: None | Response: False


In [36]:
# Manjummel Boys IMDb ID is tt27564649
r = requests.get("http://www.omdbapi.com/?i=tt27564649&apikey=13acfb9e").json()
print(r)

{'Response': 'False', 'Error': 'Request limit reached!'}
